# Data preparation

Download, inspect, clean, and align Dolly-15k and FreshQA. The datasets remain separate.

## Setup

Select the project `.venv` kernel. See the README for installation commands.

In [1]:
from pathlib import Path
import sys
import json
import csv
import hashlib
from datetime import datetime, timezone, timedelta

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "README.md").is_file() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook inside a copy of this project.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select this project's .venv kernel before running data operations.")
RUN_DATE = datetime.now(timezone(timedelta(hours=5.5))).date().isoformat()
print("Project venv verified:", PROJECT_ROOT.name)
print("Python version:", sys.version.split()[0])
print("Audit date (IST):", RUN_DATE)

Project venv verified: ASBD Project
Python version: 3.13.0
Audit date (IST): 2026-10-15


## Download

Fetch the original datasets; keep existing downloads.

In [2]:
"""Download official raw datasets without replacing existing local files."""
import csv
import io
import json
from pathlib import Path
import urllib.request

PROJECT = PROJECT_ROOT
SOURCES = {
    "databricks-dolly-15k.jsonl": "https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl?download=true",
    "freshqa_2026-04-21.csv": "https://docs.google.com/spreadsheets/d/1_8mi-yuK30mvoDJu1KQXD6ODem7MKMcIgVAwDSzJkjM/export?format=csv",
}


def download():
    raw = PROJECT / "data" / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    sources = dict(SOURCES)
    for filename, url in sources.items():
        target = raw / filename
        if target.exists():
            print("Kept existing:", filename, target.stat().st_size, "bytes")
            continue
        with urllib.request.urlopen(url, timeout=120) as response:
            payload = response.read()
        # Validate before saving, so an HTML error/login page cannot masquerade as data.
        if filename.endswith(".jsonl"):
            records = [json.loads(line) for line in payload.decode("utf-8-sig").splitlines() if line.strip()]
            if not records or not all(isinstance(row, dict) for row in records):
                raise ValueError("Invalid Dolly response")
        elif filename.endswith(".csv"):
            records = list(csv.reader(io.StringIO(payload.decode("utf-8-sig"), newline="")))
            if not any("question" in row and "id" in row for row in records):
                raise ValueError("FreshQA response has no expected header")
        with target.open("xb") as handle:
            handle.write(payload)
        print("Downloaded:", filename, len(payload), "bytes")



download()
RAW_DIR = PROJECT / 'data' / 'raw'

Kept existing: databricks-dolly-15k.jsonl 13085339 bytes
Kept existing: freshqa_2026-04-21.csv 202939 bytes


## Inspect

Check record counts, columns, and source categories.

In [3]:
from collections import Counter

dolly_raw = [json.loads(line) for line in (RAW_DIR / "databricks-dolly-15k.jsonl")
             .read_text(encoding="utf-8-sig").splitlines() if line.strip()]
with (RAW_DIR / "freshqa_2026-04-21.csv").open(encoding="utf-8-sig", newline="") as handle:
    fresh_raw_rows = list(csv.reader(handle))
fresh_header_index = next(i for i, row in enumerate(fresh_raw_rows) if "question" in row and "id" in row)
fresh_header = fresh_raw_rows[fresh_header_index]
fresh_raw = [dict(zip(fresh_header, row)) for row in fresh_raw_rows[fresh_header_index + 1:]
             if any(value.strip() for value in row)]
print("Dolly rows:", len(dolly_raw))
print("Dolly columns:", list(dolly_raw[0]))
print("Dolly categories:", dict(Counter(row["category"] for row in dolly_raw)))
print("FreshQA rows:", len(fresh_raw))
print("FreshQA header starts at CSV record:", fresh_header_index + 1)
print("FreshQA columns:", fresh_header)
print("FreshQA fact types:", dict(Counter(row["fact_type"] for row in fresh_raw)))

Dolly rows: 15011
Dolly columns: ['instruction', 'context', 'response', 'category']
Dolly categories: {'closed_qa': 1773, 'classification': 2136, 'open_qa': 3742, 'information_extraction': 1506, 'brainstorming': 1766, 'general_qa': 2191, 'summarization': 1188, 'creative_writing': 709}
FreshQA rows: 600
FreshQA header starts at CSV record: 3
FreshQA columns: ['id', 'split', 'question', 'effective_year', 'next_review', 'false_premise', 'num_hops', 'fact_type', 'source', 'answer_0', 'answer_1', 'answer_2', 'answer_3', 'answer_4', 'answer_5', 'answer_6', 'answer_7', 'answer_8', 'answer_9', 'note']
FreshQA fact types: {'slow-changing': 220, 'never-changing': 225, 'fast-changing': 155}


## Clean

Normalize Unicode and line endings, remove fully identical records, and flag review cases.

In [4]:
"""Conservative, reproducible dataset cleaning using only the standard library."""
import argparse
from collections import Counter, defaultdict
import csv
from datetime import datetime, timedelta, timezone
import hashlib
import io
import json
from pathlib import Path
import unicodedata

PROJECT = PROJECT_ROOT
DOLLY_FIELDS = ["instruction", "context", "response", "category"]
FRESH_FIELDS = ["id", "split", "question", "effective_year", "next_review",
                "false_premise", "num_hops", "fact_type", "source",
                *[f"answer_{i}" for i in range(10)], "note"]
CATEGORIES = {"closed_qa", "classification", "open_qa", "information_extraction",
              "brainstorming", "general_qa", "summarization", "creative_writing"}


def digest(content):
    return hashlib.sha256(content).hexdigest()


def normalize(value):
    """Preserve meaningful spacing/indentation and paragraphs; no token filtering."""
    value = unicodedata.normalize("NFC", value.replace("\r\n", "\n").replace("\r", "\n"))
    return "" if not value.strip() else value


def fingerprint(row, fields):
    # Review grouping only, never a reason to delete differently answered prompts.
    values = [" ".join(row[field].casefold().split()) for field in fields]
    return digest(json.dumps(values, ensure_ascii=False).encode("utf-8"))

In [5]:
def clean(records, fields, request_field, context_fields, metadata_fields, dataset):
    kept, provenance, review = [], [], []
    seen, groups = {}, defaultdict(list)
    counts = Counter()
    empty_before = Counter()
    for source_row, row, parse_error in records:
        counts["input_records"] += 1
        if parse_error or not isinstance(row, dict) or any(
            field not in row or not isinstance(row[field], str) for field in fields
        ):
            counts["invalid_records_excluded"] += 1
            review.append({"source_row": source_row, "cleaned_row": "",
                           "issue": "invalid_record_excluded", "detail": parse_error or "invalid field schema"})
            continue
        if set(row) != set(fields):
            raise ValueError(f"Unexpected {dataset} columns; review the schema before cleaning.")
        empty_before.update(field for field in fields if not row[field].strip())
        cleaned = {field: normalize(row[field]) for field in fields}
        for field in metadata_fields:
            cleaned[field] = cleaned[field].strip()
        changed = [field for field in fields if cleaned[field] != row[field]]
        counts["normalized_fields"] += len(changed)
        counts["normalized_records"] += bool(changed)
        if not cleaned[request_field].strip():
            counts["empty_requests_excluded"] += 1
            review.append({"source_row": source_row, "cleaned_row": "",
                           "issue": "empty_request_excluded", "detail": "request is whitespace-only"})
            continue
        key = json.dumps(cleaned, sort_keys=True, ensure_ascii=False)
        if key in seen:
            counts["identical_records_removed"] += 1
            review.append({"source_row": source_row, "cleaned_row": seen[key],
                           "issue": "identical_record_removed", "detail": "all source fields match retained record"})
            continue
        cleaned_row = len(kept) + 1
        seen[key] = cleaned_row
        kept.append(cleaned)
        group_id = fingerprint(cleaned, context_fields)
        groups[group_id].append(cleaned_row)
        provenance.append({"cleaned_row": cleaned_row, "source_row": source_row,
                           "original_id": cleaned.get("id", ""), "input_group_id": group_id,
                           "normalized_fields": ",".join(changed)})
        issues = []
        if len(cleaned[request_field].split()) < 2:
            issues.append(("short_request_review", "fewer than two whitespace-separated words; retained"))
        if any("\ufffd" in value for value in cleaned.values()):
            issues.append(("replacement_character_review", "Unicode replacement character present; retained"))
        if any(unicodedata.category(char) == "Cc" and char not in "\n\t"
               for value in cleaned.values() for char in value):
            issues.append(("control_character_review", "unusual control character present; retained"))
        if dataset == "dolly" and cleaned["category"] not in CATEGORIES:
            issues.append(("unexpected_category_review", cleaned["category"]))
        if dataset == "freshqa":
            expected = {"split": {"DEV", "TEST"}, "false_premise": {"TRUE", "FALSE"},
                        "num_hops": {"one-hop", "multi-hop"},
                        "fact_type": {"never-changing", "slow-changing", "fast-changing"}}
            for field, values in expected.items():
                if cleaned[field] not in values:
                    issues.append(("unexpected_metadata_review", field + ": " + cleaned[field]))
            if not cleaned["answer_0"].strip():
                issues.append(("empty_reference_answer_review", "answer_0 is empty; retained"))
        for issue, detail in issues:
            review.append({"source_row": source_row, "cleaned_row": cleaned_row,
                           "issue": issue, "detail": detail})
    repeated = {key: values for key, values in groups.items() if len(values) > 1}
    for group_id, members in repeated.items():
        for member in members:
            review.append({"source_row": provenance[member - 1]["source_row"],
                           "cleaned_row": member, "issue": "repeated_input_review",
                           "detail": group_id + "; cleaned rows " + ",".join(map(str, members))})
    counts["output_records"] = len(kept)
    counts["repeated_input_groups_retained"] = len(repeated)
    counts["records_in_repeated_input_groups"] = sum(map(len, repeated.values()))
    counts["review_entries"] = len(review)
    for field in ("invalid_records_excluded", "empty_requests_excluded", "identical_records_removed"):
        counts.setdefault(field, 0)
    assert counts["input_records"] == (len(kept) + counts["invalid_records_excluded"]
                                        + counts["empty_requests_excluded"]
                                        + counts["identical_records_removed"])
    audit = {**dict(counts), "empty_fields_before": dict(empty_before),
             "empty_fields_after": {field: sum(not row[field].strip() for row in kept) for field in fields},
             "review_issue_counts": dict(Counter(row["issue"] for row in review))}
    return kept, provenance, review, audit

In [6]:
def csv_text(rows, fields):
    handle = io.StringIO(newline="")
    writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return handle.getvalue()


def write_text(path, content):
    with path.open("w", encoding="utf-8", newline="") as handle:
        handle.write(content)




def clean_sources(record_date=None):
    args = argparse.Namespace(record_date=record_date)
    raw = PROJECT / "data" / "raw"
    output = PROJECT / "data" / "cleaned"
    if not output.resolve().is_relative_to(PROJECT / "data") or output.resolve() == raw.resolve():
        raise ValueError("Output must be a separate project data directory.")
    inputs = {"dolly": raw / "databricks-dolly-15k.jsonl",
              "freshqa": raw / "freshqa_2026-04-21.csv"}
    originals = {name: path.read_bytes() for name, path in inputs.items()}
    dolly_records = []
    for source_row, line in enumerate(originals["dolly"].decode("utf-8-sig").splitlines(), 1):
        if not line.strip():
            continue
        try:
            dolly_records.append((source_row, json.loads(line), None))
        except json.JSONDecodeError as exc:
            dolly_records.append((source_row, None, str(exc)))
    csv_rows = list(csv.reader(io.StringIO(originals["freshqa"].decode("utf-8-sig"), newline="")))
    header_index = next(i for i, row in enumerate(csv_rows) if row == FRESH_FIELDS)
    fresh_records = []
    blank_rows = 0
    for source_row, row in enumerate(csv_rows[header_index + 1:], header_index + 2):
        if not any(value.strip() for value in row):
            blank_rows += 1
            continue
        fresh_records.append((source_row, dict(zip(FRESH_FIELDS, row)),
                              None if len(row) == len(FRESH_FIELDS) else "incorrect CSV field count"))
    jobs = {
        "dolly": (dolly_records, DOLLY_FIELDS, "instruction", ["instruction", "context"], ["category"]),
        "freshqa": (fresh_records, FRESH_FIELDS, "question", ["question"],
                    ["id", "split", "effective_year", "next_review", "false_premise", "num_hops", "fact_type"]),
    }
    output.mkdir(parents=True, exist_ok=True)
    audit = {"record_date": args.record_date or datetime.now(timezone(timedelta(hours=5.5))).date().isoformat(),
             "scope": "cleaning only; no integration, selection, labelling, splitting or training",
             "datasets": {}}
    for name, parameters in jobs.items():
        kept, provenance, review, summary = clean(*parameters, dataset=name)
        filename = "databricks-dolly-15k.cleaned.jsonl" if name == "dolly" else "freshqa_2026-04-21.cleaned.csv"
        destination = output / filename
        if name == "dolly":
            content = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in kept)
            assert [json.loads(line) for line in content.splitlines()] == kept
        else:
            content = csv_text(kept, FRESH_FIELDS)
            assert list(csv.DictReader(io.StringIO(content, newline=""))) == kept
            summary["introductory_rows_removed"] = header_index
            summary["blank_data_rows_removed"] = blank_rows
            summary["id_collision_groups"] = sum(count > 1 for count in Counter(row["id"] for row in kept).values())
            summary["metadata_distributions"] = {field: dict(Counter(row[field] for row in kept))
                                                  for field in ("split", "fact_type", "false_premise", "num_hops")}
        write_text(destination, content)
        write_text(output / f"{name}.provenance.csv", csv_text(provenance,
                   ["cleaned_row", "source_row", "original_id", "input_group_id", "normalized_fields"]))
        write_text(output / f"{name}.review.csv", csv_text(review,
                   ["source_row", "cleaned_row", "issue", "detail"]))
        if name == "dolly":
            assert [json.loads(line) for line in destination.read_text(encoding="utf-8").splitlines()] == kept
            summary["metadata_distributions"] = {"category": dict(Counter(row["category"] for row in kept))}
        else:
            with destination.open(encoding="utf-8", newline="") as handle:
                assert list(csv.DictReader(handle)) == kept
        summary.update({"raw_file": inputs[name].relative_to(PROJECT).as_posix(),
                        "cleaned_file": destination.relative_to(PROJECT).as_posix(),
                        "raw_sha256": digest(originals[name]),
                        "cleaned_sha256": digest(destination.read_bytes())})
        audit["datasets"][name] = summary
    for name, path in inputs.items():
        assert path.read_bytes() == originals[name], "Raw source changed during cleaning"
    audit["raw_files_unchanged"] = True
    write_text(output / "cleaning_audit.json", json.dumps(audit, indent=2, ensure_ascii=False) + "\n")
    print(json.dumps(audit, indent=2, ensure_ascii=True))

In [7]:
# Hand-checkable checks for the important cleaning decisions.
assert normalize("line one\r\n    code\rline three") == "line one\n    code\nline three"
assert normalize("e\u0301") == "\u00e9"
assert normalize(" \t\n") == ""
sample = {"instruction": "Explain this example", "context": "", "response": "First answer", "category": "general_qa"}
alternative = {**sample, "response": "Another answer"}
empty = {**sample, "instruction": "  "}
kept, _, _, check_audit = clean([(1, sample, None), (2, dict(sample), None),
                                (3, alternative, None), (4, empty, None)],
                               DOLLY_FIELDS, "instruction", ["instruction", "context"], ["category"], "dolly")
assert len(kept) == 2 and kept[1]["response"] == "Another answer"
assert check_audit["identical_records_removed"] == 1
assert check_audit["empty_requests_excluded"] == 1
assert kept[0]["context"] == ""
quoted = [{"a": 'a comma, then a quote "', "b": "line one\nline two"}]
assert list(csv.DictReader(io.StringIO(csv_text(quoted, ["a", "b"]), newline=""))) == quoted
print("Cleaning checks passed.")

Cleaning checks passed.


In [8]:
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
expected_cleaned = ["databricks-dolly-15k.cleaned.jsonl", "freshqa_2026-04-21.cleaned.csv",
                    "dolly.provenance.csv", "dolly.review.csv", "freshqa.provenance.csv",
                    "freshqa.review.csv", "cleaning_audit.json"]
present = [(CLEANED_DIR / filename).exists() for filename in expected_cleaned]
if any(present) and not all(present):
    raise RuntimeError("Partial cleaned output exists. Inspect it before continuing; nothing was overwritten.")
if all(present):
    print("Kept existing cleaned datasets and audit.")
else:
    clean_sources(record_date=RUN_DATE)
cleaning_audit = json.loads((CLEANED_DIR / "cleaning_audit.json").read_text(encoding="utf-8"))
for name, info in cleaning_audit["datasets"].items():
    assert hashlib.sha256((PROJECT_ROOT / info["raw_file"]).read_bytes()).hexdigest() == info["raw_sha256"]
    assert hashlib.sha256((PROJECT_ROOT / info["cleaned_file"]).read_bytes()).hexdigest() == info["cleaned_sha256"]
    print(name, ":", info["input_records"], "->", info["output_records"], "records")

Kept existing cleaned datasets and audit.
dolly : 15011 -> 15003 records
freshqa : 600 -> 600 records


## Align columns

Both tables use `id`, `prompt`, `user_context`, `route`, and `label_reason`.
Answers and source metadata stay in companion files. Annotation fields start blank.

In [9]:
"""Align and prune columns in separate datasets; do not merge or assign routes."""
import argparse
from collections import defaultdict
import csv
from datetime import datetime, timedelta, timezone
import hashlib
import io
import json
from pathlib import Path

PROJECT = PROJECT_ROOT
FIELDS = ["id", "prompt", "user_context", "route", "label_reason"]


def csv_content(rows):
    stream = io.StringIO(newline="")
    writer = csv.DictWriter(stream, fieldnames=FIELDS, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue()


def prepare(dataset, rows, provenance, review):
    if len(rows) != len(provenance):
        raise ValueError("Cleaned records and provenance have different lengths")
    flags = defaultdict(list)
    for issue in review:
        if issue["cleaned_row"] and issue["issue"] != "identical_record_removed":
            flags[int(issue["cleaned_row"])].append(issue["issue"])
    primary, metadata = [], []
    for index, (row, origin) in enumerate(zip(rows, provenance, strict=True), 1):
        if int(origin["cleaned_row"]) != index:
            raise ValueError("Provenance row order does not match cleaned data")
        source_id = origin["source_row"] if dataset == "dolly" else row["id"]
        identifier = f"{dataset}-{source_id}"
        prompt = row["instruction"] if dataset == "dolly" else row["question"]
        context = row["context"] if dataset == "dolly" else ""
        if not prompt.strip():
            raise ValueError("A source request is empty")
        primary.append({"id": identifier, "prompt": prompt, "user_context": context,
                        "route": "", "label_reason": ""})
        record = {"id": identifier, "source_dataset": dataset, "original_id": row.get("id"),
                  "source_row": int(origin["source_row"]),
                  "source_input_group_id": origin["input_group_id"],
                  "cleaning_flags": sorted(set(flags[index]))}
        if dataset == "dolly":
            record.update({"reference_answers": [row["response"]],
                           "source_metadata": {"category": row["category"]}})
        else:
            record.update({"reference_answers": [row[f"answer_{n}"] for n in range(10)
                                                 if row[f"answer_{n}"].strip()],
                           "source_metadata": {key: row[key] for key in
                                               ("split", "effective_year", "next_review", "false_premise",
                                                "num_hops", "fact_type", "source")}})
            if row["note"].strip():
                # Current snapshot has no notes; preserve unexpected future notes.
                record["source_metadata"]["note"] = row["note"]
        metadata.append(record)
    ids = [row["id"] for row in primary]
    if len(set(ids)) != len(ids):
        raise ValueError("Prepared IDs are not unique")
    return primary, metadata

In [10]:
def prepare_separate_tables(record_date=None):
    args = argparse.Namespace(record_date=record_date)
    cleaned = PROJECT / "data" / "cleaned"
    output = PROJECT / "data" / "prepared"
    if not output.resolve().is_relative_to(PROJECT / "data"):
        raise ValueError("Output is outside the project data directory")
    filenames = [f"{name}.{suffix}" for name in ("dolly", "freshqa")
                 for suffix in ("annotation.csv", "metadata.jsonl")]
    filenames.append("column_preparation_audit.json")
    if any((output / name).exists() for name in filenames):
        raise FileExistsError("Prepared files already exist. Refusing to overwrite possible annotations.")
    source_paths = {
        "dolly": cleaned / "databricks-dolly-15k.cleaned.jsonl",
        "freshqa": cleaned / "freshqa_2026-04-21.cleaned.csv",
    }
    tracked = [*source_paths.values(), *[cleaned / f"{name}.{suffix}.csv"
               for name in ("dolly", "freshqa") for suffix in ("provenance", "review")]]
    before = {path: path.read_bytes() for path in tracked}
    with source_paths["freshqa"].open(encoding="utf-8", newline="") as handle:
        fresh = list(csv.DictReader(handle))
    sources = {"dolly": [json.loads(line) for line in before[source_paths["dolly"]]
                         .decode("utf-8").splitlines() if line.strip()], "freshqa": fresh}
    date = args.record_date or datetime.now(timezone(timedelta(hours=5.5))).date().isoformat()
    audit = {"record_date": date, "scope": "column alignment/pruning only; no merge, row selection or route labels",
             "shared_columns": FIELDS, "prediction_inputs": ["prompt", "user_context"], "datasets": {}}
    pending = {}
    all_ids = set()
    for name, rows in sources.items():
        with (cleaned / f"{name}.provenance.csv").open(encoding="utf-8", newline="") as handle:
            provenance = list(csv.DictReader(handle))
        with (cleaned / f"{name}.review.csv").open(encoding="utf-8", newline="") as handle:
            review = list(csv.DictReader(handle))
        primary, metadata = prepare(name, rows, provenance, review)
        ids = {row["id"] for row in primary}
        assert not all_ids & ids, "IDs overlap across datasets"
        all_ids.update(ids)
        content = csv_content(primary)
        assert list(csv.DictReader(io.StringIO(content, newline=""))) == primary
        assert len(primary) == len(rows) == len(metadata)
        assert all(not row["route"] and not row["label_reason"] for row in primary)
        assert all(list(row) == FIELDS for row in primary)
        assert [row["id"] for row in primary] == [row["id"] for row in metadata]
        # Assert the prediction text was not rewritten or confused with an answer.
        for source, prepared in zip(rows, primary, strict=True):
            assert prepared["prompt"] == source["instruction" if name == "dolly" else "question"]
            assert prepared["user_context"] == (source["context"] if name == "dolly" else "")
        pending[f"{name}.annotation.csv"] = content
        pending[f"{name}.metadata.jsonl"] = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in metadata)
        audit["datasets"][name] = {"source_columns": list(rows[0]), "source_rows": len(rows),
                                   "prepared_rows": len(primary), "blank_route_labels": len(primary),
                                   "metadata_records": len(metadata), "id_uniqueness_checked": True,
                                   "prompt_and_context_preserved": True}
    for path, content in before.items():
        assert path.read_bytes() == content, "An input file changed"
    audit["inputs_unchanged"] = True
    audit["input_sha256"] = {path.relative_to(PROJECT).as_posix(): hashlib.sha256(content).hexdigest()
                              for path, content in before.items()}
    audit["output_sha256"] = {name: hashlib.sha256(content.encode("utf-8")).hexdigest()
                               for name, content in pending.items()}
    pending["column_preparation_audit.json"] = json.dumps(audit, indent=2) + "\n"
    output.mkdir(parents=True, exist_ok=True)
    for filename, content in pending.items():
        with (output / filename).open("x", encoding="utf-8", newline="") as handle:
            handle.write(content)
    for name in sources:
        with (output / f"{name}.annotation.csv").open(encoding="utf-8", newline="") as handle:
            assert len(list(csv.DictReader(handle))) == len(sources[name])
        actual_metadata = [json.loads(line) for line in (output / f"{name}.metadata.jsonl")
                           .read_text(encoding="utf-8").splitlines()]
        assert len(actual_metadata) == len(sources[name])
    print(json.dumps(audit, indent=2))

In [11]:
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
expected_prepared = ["dolly.annotation.csv", "dolly.metadata.jsonl", "freshqa.annotation.csv",
                     "freshqa.metadata.jsonl", "column_preparation_audit.json"]
present = [(PREPARED_DIR / filename).exists() for filename in expected_prepared]
if any(present) and not all(present):
    raise RuntimeError("Partial prepared output exists. Inspect it first; nothing was overwritten.")
if all(present):
    print("Kept existing annotation files: possible human labels will not be overwritten.")
else:
    prepare_separate_tables(record_date=RUN_DATE)

Kept existing annotation files: possible human labels will not be overwritten.


## Validate

Check row counts, text preservation, IDs, metadata joins, and allowed route values.

In [12]:
with (CLEANED_DIR / "freshqa_2026-04-21.cleaned.csv").open(encoding="utf-8", newline="") as handle:
    cleaned_fresh = list(csv.DictReader(handle))
cleaned_sources = {
    "dolly": [json.loads(line) for line in (CLEANED_DIR / "databricks-dolly-15k.cleaned.jsonl")
              .read_text(encoding="utf-8").splitlines()],
    "freshqa": cleaned_fresh,
}
all_prepared_ids = set()
for name, original in cleaned_sources.items():
    with (PREPARED_DIR / f"{name}.annotation.csv").open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        assert reader.fieldnames == FIELDS
        prepared = list(reader)
    metadata = [json.loads(line) for line in (PREPARED_DIR / f"{name}.metadata.jsonl")
                .read_text(encoding="utf-8").splitlines()]
    assert len(prepared) == len(original) == len(metadata)
    identifiers = [row["id"] for row in prepared]
    assert len(set(identifiers)) == len(identifiers)
    assert not all_prepared_ids.intersection(identifiers)
    all_prepared_ids.update(identifiers)
    assert identifiers == [row["id"] for row in metadata]
    for old, new in zip(original, prepared, strict=True):
        assert new["prompt"] == old["instruction" if name == "dolly" else "question"]
        assert new["user_context"] == (old["context"] if name == "dolly" else "")
        assert new["route"] in {"", "rag", "internet", "direct"}
    print(name, ":", len(prepared), "rows;", sum(bool(row["route"]) for row in prepared), "route labels assigned")
    for row in prepared[:2]:
        print({"id": row["id"], "prompt_preview": row["prompt"][:100], "route": row["route"]})
print("Validation passed. No datasets were merged and no route labels were generated.")

dolly : 15003 rows; 0 route labels assigned
{'id': 'dolly-1', 'prompt_preview': 'When did Virgin Australia start operating?', 'route': ''}
{'id': 'dolly-2', 'prompt_preview': 'Which is a species of fish? Tope or Rope', 'route': ''}
freshqa : 600 rows; 0 route labels assigned
{'id': 'freshqa-0', 'prompt_preview': 'What is the name of the first animal to land on the moon?', 'route': ''}
{'id': 'freshqa-1', 'prompt_preview': "What is the name of Leonardo DiCaprio's third child?", 'route': ''}
Validation passed. No datasets were merged and no route labels were generated.


## Output

Separate annotation tables are in `data/prepared/`. No routes have been assigned.